# 5.5 怎麼獨立控制權重衰減？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：旋鈕往哪邊轉，猜錯的代價會下降？**

藍線是 (w−3)²；橘點到綠點只走一小步。

這張圖保留靜態標註；先核對箭頭、位置與每個數字的意思。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/gradient.svg")))

**先想一想：**沒有任務梯度時，參數會保持原值嗎？

AdamW把權重衰減獨立於Adam的梯度縮放。即使loss梯度為0，只要有梯度tensor，weight decay仍可讓權重縮小。

**把直覺寫成數學：**w←(1−ηλ)w−η·Adam方向；λ是衰減強度。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
w = nn.Parameter(torch.tensor([2.0]))
opt = torch.optim.AdamW([w], lr=0.1, weight_decay=0.2)
w.grad = torch.zeros_like(w)
opt.step()
print(w.detach())
assert torch.allclose(w.detach(), torch.tensor([1.96]))

**如何讀結果：**這是一個參數的一步驗算；grad=None與grad=0不是同一種optimizer狀態。

**只改一件事：**只把weight_decay改成0。
